<a href="https://colab.research.google.com/github/abyanrizz/practicallinearalgebra/blob/main/Chapter_11_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#chapter 11
The universe is a really big and really complicated place. All animals on Earth have
a natural curiousity to explore and try to understand their environment, but we
humans are privileged with the intelligence to develop scientific and statistical tools
to take our curiousity to the next level. That’s why we have airplanes, MRI machines,
rovers on Mars, vaccines, and, of course, books like this one.
How do we understand the universe? By developing mathematically grounded theo‐
ries, and by collecting data to test and improve those theories. And this brings us to
statistical models. A statistical model is a simplified mathematical representation of
some aspect of the world. Some statistical models are simple (e.g., predicting that the
stock market will increase over decades); others are much more sophisticated, like
the Blue Brain Project that simulates brain activity with such exquisite detail that one
second of simulated activity requires 40 minutes of computation time.
A key distinction of statistical models (as opposed to other mathematical models) is
that they contain free parameters that are fit to data. For example, I know that the
stock market will go up over time, but I don’t know by how much. Therefore, I allow
the change in stock market price over time (that is, the slope) to be a free parameter
whose numerical value is determined by data.
Crafting a statistical model can be difficult and requires creativity, experience, and
expertise. But finding the free parameters based on fitting the model to data is a
simple matter of linear algebra—in fact, you already know all the math you need for
this chapter; it’s just a matter of putting the pieces together and learning the statistics
terminology.

175

General Linear Models
A statistical model is a set of equations that relates predictors (called independent
variables) to observations (called the dependent variable). In the model of the stock
market, the independent variable is time and the dependent variable is stock market
price (e.g., quantified as the S&P 500 index).
In this book I will focus on general linear models, which are abbreviated as GLM.
Regression is a type of GLM, for example.
Terminology
Statisticians use slightly different terminology than do linear algebraticians.
Table 11-1 shows the key letters and descriptions for vectors and matrices used in
the GLM.
Setting up a GLM involves (1) defining an equation that relates the predictor vari‐
ables to the dependent variable, (2) mapping the observed data onto the equations,
(3) transforming the series of equations into a matrix equation, and (4) solving that
equation.
I’ll use a simple example to make the procedure concrete. I have a model that predicts
adult height based on weight and on parents’ height. The equation looks like this:
y = β0 + β1w + β2h + ε
y is the height of an individual, w is their weight, and h is their parents’ height (the aver‐
age of mother and father). ε is an error term (also called a residual), because we cannot
reasonably expect that weight and parents’ height perfectly determines an individual’s
height; there are myriad factors that our model does not account for, and the variance
not attributable to weight and parents’ height will be absorbed by the residual.
My hypothesis is that weight and parents’ height are important for an individual’s
height, but I don’t know how important each variable is. Enter the β terms: they are

176 | Chapter 11: General Linear Models and Least Squares

the coefficients, or weights, that tell me how to combine weight and parents’ height to
predict an individual’s height. In other words, a linear weighted combination, where
the βs are the weights.
β0
is called an intercept (sometimes called a constant). The intercept term is a vector
of all 1s. Without an intercept term, the best-fit line is forced to pass through the
origin. I’ll explain why, and show a demonstration, toward the end of the chapter.
Now we have our equation, our model of the universe (well, one tiny part of it).
Next, we need to map the observed data onto the equations. For simplicity, I’m going
to make up some data in Table 11-2 (you may imagine that y and h have units of
centimeters and w has units of kilograms).
Table 11-2. Made-up data for our statistical model of height
y w h
175 70 177
181 86 190
159 63 180
165 62 172
Mapping the observed data onto our statistical model involves replicating the equa‐
tion four times (corresponding to four observations in our dataset), each time replac‐
ing the variables y, w, and h with the measured data:
175 = β0 + 70β1 + 177β2
181 = β0 + 86β1 + 190β2
159 = β0 + 63β1 + 180β2
165 = β0 + 62β1 + 172β2
I’m omitting the ε term for now; I’ll have more to say about the residuals later. We
now need to translate this system of equations into a matrix equation. I know that
you know how to do that, so I’ll print out the equation here only so you can confirm
what you already know from Chapter 10:
1 70 177
1 86 190
1 63 180
1 62 172
β0
β1
β2
=
175
181
159
165

And, of course, we can express this equation succinctly as Xβ = y.

General Linear Models | 177

1 Admittedly, I’ve never seen this equation on a Tajikistani billboard, but the point is to keep an open mind.
Solving GLMs
I’m sure you already know the main idea of this section: to solve for the vector
of unknown coefficients β, simply left-multiply both sides of the equation by the
left-inverse of X, the design matrix. The solution looks like this:

Xβ = y
X
TX
−1
X
TXβ = X
TX
−1
X
T
y
β = X
TX
−1
X
T
y

Please stare at that final equation until it is permanently tattooed into your brain.
It is called the least squares solution and is one of the most important mathematical
equations in applied linear algebra. You’ll see it in research publications, textbooks,
blogs, lectures, docstrings in Python functions, billboards in Tajikistan,1
and many
other places. You might see different letters, or possibly some additions, like the
following:
b = H
TWH + λLT
L
−1
H
T
x

The meaning of that equation and the interpretation of the additional matrices are
not important (they are various ways of regularizing the model fitting); what is
important is that you are able to see the least squares formula embedded in that
complicated-looking equation (for example, imagine setting W = I and λ = 0).
The least squares solution via the left-inverse can be translated directly into Python
code (variable X is the design matrix and variable y is the data vector):
X_leftinv = np.linalg.inv(X.T@X) @ X.T
# solve for the coefficients
beta = X_leftinv @ y
I will show results from these models—and how to interpret them—later in this
chapter; for now I’d like you to focus on how the math formulas are translated into
Python code.

178 | Chapter 11: General Linear Models and Least Squares

Left-Inverse Versus NumPy’s Least Squares Solver
The code in this chapter is a direct translation of the math into
Python code. Explicitly computing the left-inverse is not the most
numerically stable way to solve the GLM (although it is accurate
for the simple problems in this chapter), but I want you to see that
the seemingly abstract linear algebra really works. There are more
numerically stable ways to solve the GLM, including QR decomposi‐
tion (which you will see later in this chapter) and Python’s more
numerically stable methods (which you will see in the next chapter).

Is the Solution Exact?
The equation Xβ = y is exactly solvable when y is in the column space of the design
matrix X. So the question is whether the data vector is guaranteed to be in the
column space of the statistical model. The answer is no, there is no such guarantee. In
fact, the data vector y is almost never in the column space of X.
To understand why not, let’s imagine a survey of university students in which the
researchers are trying to predict average GPA (grade point average) based on drink‐
ing behavior. The survey may contain data from two thousand students yet have only
three questions (e.g., how much alcohol do you consume; how often do you black
out; what is your GPA). The data is contained in a 2000 × 3 table. The column space
of the design matrix is a 2D subspace inside that 2000D ambient dimensionality, and
the data vector is a 1D subspace inside that same ambient dimensionality.
If the data is in the column space of the design matrix, it means that the model

accounts for 100% of the variance in the data. But this almost never happens: real-
world data contains noise and sampling variability, and the models are simplifications

that do not account for all of the variability (e.g., GPA is determined by myriad
factors that our model ignores).
The solution to this conundrum is that we modify the GLM equation to allow for
a discrepancy between the model-predicted data and the observed data. It can be
expressed in several equivalent (up to a sign) ways:
Xβ = y + ε
Xβ + ε = y
ε = Xβ − y
The interpretation of the first equation is that ε is a residual, or an error term, that
you add to the data vector so that it fits inside the column space of the design matrix.
The interpretation of the second equation is that the residual term is an adjustment
to the design matrix so that it fits the data perfectly. Finally, the interpretation

Solving GLMs | 179

of the third equation is that the residual is defined as the difference between the
model-predicted data and the observed data.
There is another, very insightful, interpretation, which approaches GLMs and least
squares from a geometric perspective. I’ll get back to this in the next section.
The point of this section is that the observed data is almost never inside the subspace
spanned by the regressors. For this reason, it is also common to see the GLM
expressed as X = βy where y = y + ε.
Therefore, the goal of the GLM is to find the linear combination of the regressors that
gets as close as possible to the observed data. More on this point later; I now want to
introduce you to the geometric perspective of least squares.
A Geometric Perspective on Least Squares
So far I’ve introduced the solution to the GLM from the algebraic perspective of
solving a matrix equation. There is also a geometric perspective to the GLM, which
provides an alternative perspective and helps reveal several important features of the
least squares solution.
Let’s consider that the column space of the design matrix C X is a subspace of R
M.
It’s typically a very low-dimensional subspace (that is, N << M), because statistical
models tend to have many more observations (rows) than predictors (columns). The
dependent variable is a vector y ∈ R

M. The questions at hand are, is the vector y in

the column space of the design matrix, and if not, what coordinate inside the column
space of the design matrix is as close as possible to the data vector?
The answer to the first question is no, as I discussed in the previous section. The
second question is profound, because you already learned the answer in Chapter 2.
Consider Figure 11-1 while thinking about the solution.

Figure 11-1. The abstracted geometric view of GLM: find the point in the column space
of the design matrix that is as close as possible to the data vector
180 | Chapter 11: General Linear Models and Least Squares

So, our goal is to find the set of coefficients β such that the weighted combination
of columns in X minimizes the distance to data vector y. We can call that projection
vector ε. How do we find the vector ε and the coefficients β? We use orthogonal
vector projection, just like what you learned in Chapter 2. This means we can apply
the same approach as in Chapter 2, but using matrices instead of vectors. The key
insight is that the shortest distance between y and X is given by the projection vector
y − Xβ that meets X at a right angle:

X
T
ε = 0
X
T
y − Xβ = 0
X
T
y − X
TXβ = 0
X
TXβ = X
T
y
β = X
TX
−1
X
T
y

That progression of equations is remarkable: we started from thinking about the
GLM as a geometric projection of a data vector onto the column space of a design
matrix, applied the principle of orthogonal vector projection that you learned about
early on in the book, and voilà! We have rederived the same left-inverse solution that
we got from the algebraic approach.
Why Does Least Squares Work?
Why is it called “least squares”? What are these so-called squares, and why does this
method give us the least of them?
The “squares” here refers to squared errors between the predicted data and the
observed data. There is an error term for each ith predicted data point, which is
defined as εi = Xiβ − yi

. Note that each data point is predicted using the same set
of coefficients (that is, the same weights for combining the predictors in the design
matrix). We can capture all errors in one vector:
ε = Xβ − y
If the model is a good fit to the data, then the errors should be small. Therefore, we
can say that the objective of model fitting is to choose the elements in β that minimize
the elements in ε. But just minimizing the errors would cause the model to predict
values toward negative infinity. Thus, instead we minimize the squared errors, which
correspond to their geometric squared distance to the observed data y, regardless of

Solving GLMs | 181

2 In case you were wondering, it is also possible to minimize the absolute distances instead of the squared
distances. Those two objectives can lead to different results; one advantage of squared distance is the
convenient derivative, which leads to the least squares solution.
3 If you are not comfortable with matrix calculus, then don’t worry about the equations; the point is that we
took the derivative with respect to β using the chain rule.
whether the prediction error itself is positive or negative.2

This is the same thing as
minimizing the squared norm of the errors. Hence the name “least squares.” That
leads to the following modification:
∥ ε ∥ 2
= ∥ Xβ − y ∥ 2
We can now view this as an optimization problem. In particular, we want to find
the set of coefficients that minimizes the squared errors. That minimization can be
expressed as follows:
min
β
∥ Xβ − y ∥ 2
The solution to this optimization can be found by setting the derivative of the
objective to zero and applying a bit of differential calculus3

and a bit of algebra:

0 = d
dβ ∥ Xβ − y ∥ 2
= 2X
T Xβ − y
0 = X
TXβ − X
T
y

X
TXβ = X
T
y
β = X
TX
−1
X
T
y

Amazingly enough, we started from a different perspective—minimize the squared
distance between the model-predicted values and the observed values—and again we
rediscovered the same solution to least squares that we reached simply by using our
linear algebra intuition.
Figure 11-2 shows some observed data (black squares), their model-predicted values
(gray dots), and the distances between them (gray dashed lines). All model-predicted
values lie on a line; the goal of least squares is to find the slope and intercept of that
line that minimizes the distances from predicted to observed data.